# Optimización de Rutas de Recolección de Residuos Sólidos
## Zona Los Sauces – Pimentel, Chiclayo
### Algoritmo Genético aplicado al Problema del Viajero (TSP)

---

**Curso:** Desarrollo de Sistemas Inteligentes  
**Universidad Católica Santo Toribio de Mogrovejo – USAT**

---

### Descripción del problema

Se dispone de **20 puntos de recolección de residuos sólidos** en la urbanización Los Sauces, Pimentel – Chiclayo. Un camión recolector debe partir desde un **punto de inicio fijo**, visitar **todos los puntos de recolección exactamente una vez** y llegar al **punto de destino fijo**, minimizando la distancia total recorrida.

Este problema se modela como una variante del **Travelling Salesman Problem (TSP) con inicio y fin fijos** (ruta abierta), y se resuelve mediante un **Algoritmo Genético (AG)** implementado desde cero en Python, sin librerías de optimización.

### Referencia metodológica

La implementación sigue el modelo propuesto por **Vega-Huerta et al. (2024)** — *Generation of Routes Using Genetic Algorithms to Optimize the Domestic Solid Waste Collection* — adaptado a las condiciones geográficas reales de la zona de estudio, con representación de rutas mediante **cromosomas de permutación** y operadores específicos para TSP descritos por **Larraña ga et al. (1999)**.

---
## CELDA 1 – Instalación de librerías auxiliares

Se instalan librerías solo para:
- Descargar la red vial real (OSMnx + NetworkX)
- Visualizar el mapa interactivo (Folium)
- Graficar la evolución del algoritmo (Matplotlib)

**El algoritmo genético NO usa ninguna de estas librerías. Está implementado con Python puro.**

In [ ]:
# ============================================================
# CELDA 1 – INSTALACIÓN DE LIBRERÍAS AUXILIARES
# ============================================================
# Estas librerías son solo para soporte: red vial, mapas y gráficos.
# El algoritmo genético se implementa íntegramente con Python estándar.

!pip install -q osmnx networkx folium pandas matplotlib

---
## CELDA 2 – Importación de módulos

Se importan los módulos necesarios. La lógica del AG solo requiere `math` y `random` de la biblioteca estándar.

In [ ]:
# ============================================================
# CELDA 2 – IMPORTACIÓN DE MÓDULOS
# ============================================================

import math
import random
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import folium

from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# Intentar importar OSMnx para usar la red vial real de OpenStreetMap.
# Si falla (sin internet o error de instalación), se usará distancia euclidiana como respaldo.
try:
    import osmnx as ox
    OSMNX_OK = True
    print("OSMnx disponible. Se usará la red vial real de OpenStreetMap.")
except Exception as e:
    OSMNX_OK = False
    print(f"OSMnx no disponible ({e}). Se usará distancia euclidiana como respaldo.")

---
## CELDA 3 – Parámetros del Algoritmo Genético

Se definen los hiperparámetros del AG, basados en los valores experimentales sugeridos por Vega-Huerta et al. (2024) y validados en Larrañaga et al. (1999) para instancias de TSP de tamaño medio.

| Parámetro | Valor | Justificación |
|---|---|---|
| Tamaño población | 100 | Suficiente diversidad para 20 nodos |
| Generaciones | 300 | Convergencia garantizada para esta escala |
| Prob. cruce (OX) | 0.90 | Alto para explotar buenas soluciones |
| Prob. mutación swap | 0.20 | Diversidad sin destruir buenas rutas |
| Prob. mutación inversión | 0.10 | Permite reordenar subtrayectos |
| Elitismo | 2 | Preserva los 2 mejores individuos |
| Torneo | 4 | Presión selectiva moderada |

In [ ]:
# ============================================================
# CELDA 3 – PARÁMETROS DEL ALGORITMO GENÉTICO
# ============================================================

# Semilla para reproducibilidad de resultados.
SEMILLA = 42
random.seed(SEMILLA)

# --- Parámetros del AG ---

# Número de individuos (rutas) en cada generación.
TAM_POBLACION = 100

# Número máximo de generaciones (iteraciones del ciclo evolutivo).
GENERACIONES = 300

# Probabilidad de aplicar cruce entre dos padres seleccionados.
PROB_CRUCE = 0.90

# Probabilidad de aplicar mutación por intercambio (swap) a un cromosoma.
PROB_MUTACION_SWAP = 0.20

# Probabilidad de aplicar mutación por inversión de subtrayecto.
PROB_MUTACION_INVERSION = 0.10

# Número de individuos élite que pasan sin cambios a la siguiente generación.
ELITISMO = 2

# Número de individuos participantes en cada torneo de selección.
TAM_TORNEO = 4

# Generaciones consecutivas sin mejora antes de detener el algoritmo.
PACIENCIA = 60

print("Parámetros cargados correctamente.")
print(f"  Población: {TAM_POBLACION} | Generaciones: {GENERACIONES} | "
      f"P_cruce: {PROB_CRUCE} | P_swap: {PROB_MUTACION_SWAP} | P_inv: {PROB_MUTACION_INVERSION}")

---
## CELDA 4 – Definición de puntos de recolección

Se definen los **20 puntos de recolección** de residuos sólidos de la zona Los Sauces – Pimentel, más el punto de inicio (depot) y el punto de destino final.

- **Punto 0**: Punto de INICIO (depot del camión recolector).
- **Puntos 1–19**: Puntos intermedios de recolección (deben visitarse todos exactamente una vez).
- **Punto 20**: Punto de FIN (destino final del recorrido).

Las coordenadas son en grados decimales (WGS84), convertidas desde las coordenadas DMS originales.

In [ ]:
# ============================================================
# CELDA 4 – PUNTOS DE RECOLECCIÓN (ZONA LOS SAUCES – PIMENTEL)
# ============================================================
# Coordenadas en grados decimales (WGS84).
# Punto 0 = INICIO (depot), Punto 20 = FIN, Puntos 1-19 = recolección.

PUNTOS = [
    {"id": 0,  "nombre": "INICIO (Depot)",        "lat": -6.7952778, "lng": -79.8843056},
    {"id": 1,  "nombre": "Punto 1",               "lat": -6.7940556, "lng": -79.8846389},
    {"id": 2,  "nombre": "Punto 2",               "lat": -6.7933611, "lng": -79.8843889},
    {"id": 3,  "nombre": "Punto 3",               "lat": -6.7922778, "lng": -79.8834444},
    {"id": 4,  "nombre": "Punto 4",               "lat": -6.7914167, "lng": -79.8835278},
    {"id": 5,  "nombre": "Punto 5",               "lat": -6.7905556, "lng": -79.8835833},
    {"id": 6,  "nombre": "Punto 6",               "lat": -6.7905833, "lng": -79.8845556},
    {"id": 7,  "nombre": "Punto 7",               "lat": -6.7909722, "lng": -79.8853611},
    {"id": 8,  "nombre": "Punto 8",               "lat": -6.7914722, "lng": -79.8848333},
    {"id": 9,  "nombre": "Punto 9",               "lat": -6.7910278, "lng": -79.8866667},
    {"id": 10, "nombre": "Punto 10",              "lat": -6.7910556, "lng": -79.8892500},
    {"id": 11, "nombre": "Punto 11",              "lat": -6.7899167, "lng": -79.8880833},
    {"id": 12, "nombre": "Punto 12",              "lat": -6.7891667, "lng": -79.8865000},
    {"id": 13, "nombre": "Punto 13",              "lat": -6.7878611, "lng": -79.8875278},
    {"id": 14, "nombre": "Punto 14",              "lat": -6.7936389, "lng": -79.8856944},
    {"id": 15, "nombre": "Punto 15",              "lat": -6.7943889, "lng": -79.8882778},
    {"id": 16, "nombre": "Punto 16",              "lat": -6.7954444, "lng": -79.8886944},
    {"id": 17, "nombre": "Punto 17",              "lat": -6.7961389, "lng": -79.8868889},
    {"id": 18, "nombre": "Punto 18",              "lat": -6.7952222, "lng": -79.8876389},
    {"id": 19, "nombre": "Punto 19",              "lat": -6.7970000, "lng": -79.8877222},
    {"id": 20, "nombre": "FIN (Destino final)",   "lat": -6.7967222, "lng": -79.8863889},
]

# Índices de inicio y fin (fijos, no forman parte del cromosoma).
IDX_INICIO = 0
IDX_FIN    = 20

# Índices de los puntos intermedios (los que el AG debe ordenar).
INTERMEDIOS = list(range(1, 20))  # [1, 2, ..., 19]

df_puntos = pd.DataFrame(PUNTOS)
print(f"Total de puntos: {len(PUNTOS)} (1 inicio + 19 intermedios + 1 fin)")
display(df_puntos)

---
## CELDA 5 – Construcción de la matriz de distancias

La **función de evaluación (fitness)** del AG requiere conocer la distancia entre cualquier par de puntos. Se construye una **matriz de distancias** usando:

1. **Red vial real de OpenStreetMap** (via OSMnx + NetworkX): calcula la distancia de camino más corto por las calles reales.
2. **Distancia euclidiana aproximada** (Haversine): se usa como respaldo si OSMnx no está disponible.

La representación en cromosoma es **independiente** de esta matriz; la matriz solo se consulta durante la evaluación del fitness.

In [ ]:
# ============================================================
# CELDA 5 – CONSTRUCCIÓN DE LA MATRIZ DE DISTANCIAS
# ============================================================

def distancia_haversine(lat1, lng1, lat2, lng2):
    """
    Calcula la distancia en metros entre dos puntos geográficos
    usando la fórmula de Haversine.
    Se usa como distancia euclidiana aproximada sobre la esfera terrestre.
    """
    R = 6371000  # Radio de la Tierra en metros.
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def construir_matriz_haversine(puntos):
    """
    Construye una matriz NxN de distancias Haversine (en metros)
    para todos los puntos definidos.
    """
    n = len(puntos)
    mat = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                mat[i][j] = distancia_haversine(
                    puntos[i]["lat"], puntos[i]["lng"],
                    puntos[j]["lat"], puntos[j]["lng"]
                )
    return mat


def construir_matriz_osm(puntos):
    """
    Construye la matriz de distancias usando la red vial real de OSM.
    Para cada par de puntos, calcula el camino más corto en metros
    sobre el grafo de calles descargado con OSMnx.
    Devuelve la matriz y el grafo G (necesario para visualizar rutas).
    """
    lats = [p["lat"] for p in puntos]
    lngs = [p["lng"] for p in puntos]
    lat_c = sum(lats) / len(lats)
    lng_c = sum(lngs) / len(lngs)

    print("Descargando red vial de OpenStreetMap...")
    # Radio de 1.5 km desde el centroide para cubrir toda la zona.
    G = ox.graph_from_point((lat_c, lng_c), dist=1500, network_type="drive")
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    print(f"Red vial descargada: {len(G.nodes)} nodos, {len(G.edges)} aristas.")

    # Snap: encontrar el nodo OSM más cercano a cada punto de recolección.
    nodos_snap = [
        ox.distance.nearest_nodes(G, p["lng"], p["lat"])
        for p in puntos
    ]

    n = len(puntos)
    mat = [[0.0] * n for _ in range(n)]

    print("Calculando distancias por red vial...")
    for i in range(n):
        for j in range(n):
            if i != j:
                try:
                    longitud = nx.shortest_path_length(
                        G, nodos_snap[i], nodos_snap[j], weight="length"
                    )
                    mat[i][j] = longitud
                except nx.NetworkXNoPath:
                    # Si no hay camino en la red, usar Haversine como fallback.
                    mat[i][j] = distancia_haversine(
                        puntos[i]["lat"], puntos[i]["lng"],
                        puntos[j]["lat"], puntos[j]["lng"]
                    )
    print("Matriz de distancias OSM construida.")
    return mat, G, nodos_snap


# ---- Construcción de la matriz ----
G_osm = None
nodos_snap = None

if OSMNX_OK:
    try:
        MATRIZ_DIST, G_osm, nodos_snap = construir_matriz_osm(PUNTOS)
        FUENTE_DIST = "Red vial OpenStreetMap (OSMnx)"
    except Exception as e:
        print(f"Error al usar OSMnx ({e}). Usando Haversine.")
        MATRIZ_DIST = construir_matriz_haversine(PUNTOS)
        FUENTE_DIST = "Distancia Haversine (respaldo)"
else:
    MATRIZ_DIST = construir_matriz_haversine(PUNTOS)
    FUENTE_DIST = "Distancia Haversine (respaldo)"

print(f"\nFuente de distancias: {FUENTE_DIST}")
print(f"Ejemplo – distancia INICIO → Punto 1: {MATRIZ_DIST[0][1]:.1f} m")

---
## CELDA 6 – Representación del cromosoma y evaluación (Fitness)

### Representación

Siguiendo la **representación de ruta (path representation)** descrita por Larrañaga et al. (1999), cada individuo es una **permutación** de los 19 puntos intermedios:

```
Cromosoma: [p₃, p₇, p₁₄, p₁, p₅, ..., p₁₂]
Ruta completa: INICIO → p₃ → p₇ → p₁₄ → p₁ → p₅ → ... → p₁₂ → FIN
```

El punto de inicio y fin son **fijos** y no forman parte del cromosoma, tal como lo implementa el paper de referencia (nodo inicial NI y nodo final NF).

### Función Fitness

La calidad de una ruta se mide como la **distancia total recorrida** (en metros). El objetivo es **minimizar** esta distancia:

$$\text{fitness}(c) = \frac{1}{d_{\text{total}}(c)}$$

Donde $d_{\text{total}}$ se calcula sumando las distancias entre nodos consecutivos de la ruta completa (inicio → intermedios → fin).

In [ ]:
# ============================================================
# CELDA 6 – REPRESENTACIÓN DEL CROMOSOMA Y FUNCIÓN FITNESS
# ============================================================

def distancia_total_ruta(cromosoma):
    """
    Calcula la distancia total (en metros) de la ruta representada
    por el cromosoma dado.

    La ruta completa es:
        INICIO (id=0) → cromosoma[0] → cromosoma[1] → ... → cromosoma[-1] → FIN (id=20)

    El cromosoma contiene solo los índices de los puntos intermedios
    (valores del 1 al 19), en el orden en que serán visitados.
    """
    ruta_completa = [IDX_INICIO] + cromosoma + [IDX_FIN]
    distancia = 0.0
    for k in range(len(ruta_completa) - 1):
        origen  = ruta_completa[k]
        destino = ruta_completa[k + 1]
        distancia += MATRIZ_DIST[origen][destino]
    return distancia


def calcular_fitness(cromosoma):
    """
    Función fitness inversamente proporcional a la distancia total.
    A menor distancia, mayor fitness (mejor solución).
    Se agrega una pequeña constante para evitar división por cero.
    """
    dist = distancia_total_ruta(cromosoma)
    return 1.0 / (dist + 1e-9)


# --- Verificación rápida ---
cromosoma_ejemplo = list(range(1, 20))  # Ruta trivial: 1, 2, ..., 19
dist_ejemplo = distancia_total_ruta(cromosoma_ejemplo)
fit_ejemplo  = calcular_fitness(cromosoma_ejemplo)

print("Verificación de la función fitness:")
print(f"  Cromosoma ejemplo (orden secuencial 1→19):")
print(f"  Ruta: INICIO → {cromosoma_ejemplo} → FIN")
print(f"  Distancia total: {dist_ejemplo:.2f} m  ({dist_ejemplo/1000:.3f} km)")
print(f"  Fitness: {fit_ejemplo:.8f}")

---
## CELDA 7 – Paso 1 del AG: Inicialización de la población

Se genera una **población inicial aleatoria** de `TAM_POBLACION` cromosomas. Cada cromosoma es una permutación aleatoria de los 19 puntos intermedios.

La población debe ser **suficientemente diversa** para que la evaluación muestre individuos con fitness distintos. Si todos tienen el mismo fitness, el AG no puede seleccionar a los mejores para la reproducción (Larrañaga et al., 1999).

In [ ]:
# ============================================================
# CELDA 7 – PASO 1: INICIALIZACIÓN DE LA POBLACIÓN
# ============================================================

def crear_cromosoma():
    """
    Crea un cromosoma aleatorio: una permutación aleatoria de los
    índices de los puntos intermedios (1 al 19).
    Representa una ruta posible entre el punto de inicio y el fin.
    """
    cromosoma = INTERMEDIOS[:]  # Copia de [1, 2, ..., 19]
    random.shuffle(cromosoma)
    return cromosoma


def crear_poblacion(tam):
    """
    Genera una población inicial de 'tam' cromosomas aleatorios.
    Cada individuo es una solución candidata (ruta) al problema.
    """
    return [crear_cromosoma() for _ in range(tam)]


# --- Generación de la población inicial ---
poblacion_inicial = crear_poblacion(TAM_POBLACION)

# Calcular fitness de cada individuo para verificar diversidad.
fitness_iniciales = [calcular_fitness(c) for c in poblacion_inicial]
dist_iniciales    = [distancia_total_ruta(c) for c in poblacion_inicial]

print(f"Población inicial creada: {len(poblacion_inicial)} individuos")
print(f"  Distancia mínima en población inicial: {min(dist_iniciales):.1f} m")
print(f"  Distancia máxima en población inicial: {max(dist_iniciales):.1f} m")
print(f"  Distancia promedio:                    {sum(dist_iniciales)/len(dist_iniciales):.1f} m")
print(f"  Diversidad verificada: {len(set(tuple(c) for c in poblacion_inicial))} permutaciones únicas de {TAM_POBLACION}")

---
## CELDA 8 – Paso 2 del AG: Selección por torneo

La **selección por torneo** elige aleatoriamente `TAM_TORNEO` individuos de la población y retorna el que tiene el **mayor fitness**. Este mecanismo permite mantener diversidad genética al no descartar totalmente a los individuos menos aptos (Larrañaga et al., 1999).

In [ ]:
# ============================================================
# CELDA 8 – PASO 2: SELECCIÓN POR TORNEO
# ============================================================

def seleccion_torneo(poblacion):
    """
    Selección por torneo:
    1. Se eligen TAM_TORNEO individuos al azar de la población.
    2. Se compara su fitness.
    3. El individuo con el fitness más alto (menor distancia) gana el torneo.

    Ventaja: mantiene variabilidad genética porque individuos no óptimos
    también pueden ganar si sus competidores son peores.
    """
    participantes = random.sample(poblacion, TAM_TORNEO)
    # Retornar el participante con mayor fitness (menor distancia).
    ganador = max(participantes, key=calcular_fitness)
    return ganador


# --- Verificación ---
padre_prueba = seleccion_torneo(poblacion_inicial)
print("Selección por torneo verificada.")
print(f"  Padre seleccionado (primeros 5 genes): {padre_prueba[:5]}...")
print(f"  Distancia de la ruta seleccionada: {distancia_total_ruta(padre_prueba):.1f} m")

---
## CELDA 9 – Paso 3 del AG: Cruce (Order Crossover – OX1)

Se implementa el operador de **cruce por orden (Order Crossover, OX1)**, propuesto por Davis (1985) y recomendado por Larrañaga et al. (1999) como uno de los más efectivos para TSP.

**Procedimiento OX1:**
1. Seleccionar dos puntos de corte aleatorios.
2. Copiar el segmento del Padre 1 al hijo.
3. Rellenar los genes restantes con los valores del Padre 2 en el orden en que aparecen, omitiendo los ya presentes.

El OX1 preserva el **orden relativo** de los nodos, que es la información más relevante para TSP.

In [ ]:
# ============================================================
# CELDA 9 – PASO 3: CRUCE (ORDER CROSSOVER – OX1)
# ============================================================

def cruce_ox1(padre1, padre2):
    """
    Order Crossover (OX1) – Davis (1985).

    Produce dos hijos a partir de dos padres preservando el orden relativo
    de los genes (puntos de recolección) de cada padre.

    Pasos:
    1. Elegir dos puntos de corte aleatorios i1 e i2 (con i1 < i2).
    2. Copiar el subtrayecto padre1[i1:i2+1] al hijo.
    3. Rellenar las posiciones restantes con los genes de padre2
       en el orden en que aparecen, saltando los ya incluidos.
    4. Repetir simétricamente para producir el segundo hijo.
    """
    n = len(padre1)

    # Seleccionar dos puntos de corte distintos.
    i1, i2 = sorted(random.sample(range(n), 2))

    def _ox(p1, p2):
        hijo = [None] * n
        # Paso 1: copiar segmento del padre 1.
        hijo[i1:i2 + 1] = p1[i1:i2 + 1]
        segmento = set(hijo[i1:i2 + 1])
        # Paso 2: rellenar desde el padre 2 en orden circular.
        pos = (i2 + 1) % n
        for gen in p2[i2 + 1:] + p2[:i2 + 1]:
            if gen not in segmento:
                hijo[pos] = gen
                segmento.add(gen)
                pos = (pos + 1) % n
        return hijo

    # Aplicar cruce solo con probabilidad PROB_CRUCE.
    if random.random() < PROB_CRUCE:
        hijo1 = _ox(padre1, padre2)
        hijo2 = _ox(padre2, padre1)
    else:
        # Sin cruce: los hijos son copias exactas de los padres.
        hijo1 = padre1[:]
        hijo2 = padre2[:]

    return hijo1, hijo2


# --- Verificación ---
p1 = list(range(1, 20))
p2 = list(range(19, 0, -1))
h1, h2 = cruce_ox1(p1, p2)

print("Cruce OX1 verificado:")
print(f"  Padre 1: {p1}")
print(f"  Padre 2: {p2}")
print(f"  Hijo 1:  {h1}")
print(f"  Hijo 2:  {h2}")
print(f"  Hijo 1 es permutación válida: {sorted(h1) == list(range(1, 20))}")
print(f"  Hijo 2 es permutación válida: {sorted(h2) == list(range(1, 20))}")

---
## CELDA 10 – Paso 4 del AG: Mutación

Se implementan **dos operadores de mutación** que trabajan en secuencia:

1. **Mutación por intercambio (Swap Mutation – EM)**: selecciona dos posiciones al azar e intercambia los genes. Simple y efectiva para explorar soluciones vecinas.

2. **Mutación por inversión (Simple Inversion Mutation – SIM)**: selecciona un subtrayecto al azar y lo invierte. Permite reordenar segmentos de ruta completos.

Ambas mutaciones garantizan que el resultado sigue siendo una **permutación válida** (sin puntos duplicados ni omitidos).

In [ ]:
# ============================================================
# CELDA 10 – PASO 4: MUTACIÓN (SWAP + INVERSIÓN)
# ============================================================

def mutacion_swap(cromosoma):
    """
    Mutación por intercambio (Exchange/Swap Mutation – EM).
    Banzhaf (1990), también conocida como 'swap mutation'.

    Selecciona dos posiciones aleatorias del cromosoma e intercambia
    los genes en esas posiciones. El resultado siempre es una
    permutación válida.
    """
    nuevo = cromosoma[:]
    if random.random() < PROB_MUTACION_SWAP:
        i, j = random.sample(range(len(nuevo)), 2)
        nuevo[i], nuevo[j] = nuevo[j], nuevo[i]  # Intercambio directo.
    return nuevo


def mutacion_inversion(cromosoma):
    """
    Mutación por inversión simple (Simple Inversion Mutation – SIM).
    Holland (1975); Grefenstette (1987).

    Selecciona dos puntos de corte aleatorios e invierte el subtrayecto
    entre ellos. Preserva la validez del cromosoma porque solo reordena
    genes ya existentes.
    """
    nuevo = cromosoma[:]
    if random.random() < PROB_MUTACION_INVERSION:
        i, j = sorted(random.sample(range(len(nuevo)), 2))
        nuevo[i:j + 1] = reversed(nuevo[i:j + 1])  # Inversión del subtrayecto.
    return nuevo


def mutar(cromosoma):
    """
    Aplica ambos operadores de mutación en secuencia.
    Primero swap, luego inversión.
    Cada uno se aplica de manera independiente según su probabilidad.
    """
    nuevo = mutacion_swap(cromosoma)
    nuevo = mutacion_inversion(nuevo)
    return nuevo


# --- Verificación ---
original = list(range(1, 20))
mutado   = mutar(original)

print("Mutación verificada:")
print(f"  Original: {original}")
print(f"  Mutado:   {mutado}")
print(f"  Sigue siendo permutación válida: {sorted(mutado) == list(range(1, 20))}")

---
## CELDA 11 – Ciclo principal del Algoritmo Genético

Se ejecuta el ciclo evolutivo completo. En cada generación:

1. **Evaluación**: calcular el fitness de todos los individuos.
2. **Elitismo**: copiar los mejores `ELITISMO` individuos directamente a la siguiente generación.
3. **Selección**: elegir padres mediante torneo.
4. **Cruce (OX1)**: generar hijos con cruce por orden.
5. **Mutación**: aplicar swap e inversión con sus respectivas probabilidades.
6. **Reemplazo**: la nueva población reemplaza a la anterior.
7. **Parada temprana**: si no hay mejora en `PACIENCIA` generaciones consecutivas, detener.

Se registra el historial de evolución para análisis posterior.

In [ ]:
# ============================================================
# CELDA 11 – CICLO PRINCIPAL DEL ALGORITMO GENÉTICO
# ============================================================

def ejecutar_ag():
    """
    Ejecuta el Algoritmo Genético completo para optimizar la ruta
    de recolección de residuos sólidos.

    Retorna:
        mejor_ruta       : cromosoma con la menor distancia encontrada.
        mejor_distancia  : distancia total en metros de esa ruta.
        historial        : DataFrame con la evolución por generación.
    """

    # --- Paso 1: Inicialización ---
    # Crear población inicial aleatoria de TAM_POBLACION individuos.
    poblacion = crear_poblacion(TAM_POBLACION)

    historial = []                       # Registro de evolución.
    mejor_distancia_global = float("inf")  # Mejor distancia vista hasta ahora.
    generaciones_sin_mejora = 0          # Contador para parada temprana.
    mejor_ruta_global = None

    for gen in range(1, GENERACIONES + 1):

        # --- Paso 2: Evaluación ---
        # Calcular el fitness de cada individuo y ordenar de mayor a menor fitness
        # (equivalente a ordenar de menor a mayor distancia).
        poblacion.sort(key=calcular_fitness, reverse=True)

        # El mejor individuo de esta generación es el primero tras ordenar.
        mejor_actual = poblacion[0]
        dist_actual  = distancia_total_ruta(mejor_actual)

        # Registrar estadísticas de esta generación.
        distancias_gen = [distancia_total_ruta(c) for c in poblacion]
        historial.append({
            "Generacion": gen,
            "Mejor distancia (m)": dist_actual,
            "Distancia promedio (m)": sum(distancias_gen) / len(distancias_gen),
            "Fitness mejor": calcular_fitness(mejor_actual)
        })

        # Actualizar el mejor global encontrado.
        if dist_actual < mejor_distancia_global:
            mejor_distancia_global = dist_actual
            mejor_ruta_global = mejor_actual[:]
            generaciones_sin_mejora = 0
        else:
            generaciones_sin_mejora += 1

        # Imprimir progreso cada 50 generaciones.
        if gen % 50 == 0 or gen == 1:
            print(f"  Gen {gen:>3}: mejor = {dist_actual:.1f} m | "
                  f"global = {mejor_distancia_global:.1f} m | "
                  f"sin mejora: {generaciones_sin_mejora}")

        # Parada temprana por estancamiento.
        if generaciones_sin_mejora >= PACIENCIA:
            print(f"\nParada temprana en generación {gen} "
                  f"(sin mejora por {PACIENCIA} generaciones consecutivas).")
            break

        # --- Paso 3 al 6: Nueva generación ---

        # Elitismo: los mejores ELITISMO individuos pasan sin cambios.
        nueva_poblacion = [ind[:] for ind in poblacion[:ELITISMO]]

        # Completar la nueva población con descendencia.
        while len(nueva_poblacion) < TAM_POBLACION:

            # Selección: elegir dos padres mediante torneo.
            padre1 = seleccion_torneo(poblacion)
            padre2 = seleccion_torneo(poblacion)

            # Cruce OX1: generar dos hijos.
            hijo1, hijo2 = cruce_ox1(padre1, padre2)

            # Mutación: aplicar swap e inversión a cada hijo.
            nueva_poblacion.append(mutar(hijo1))
            if len(nueva_poblacion) < TAM_POBLACION:
                nueva_poblacion.append(mutar(hijo2))

        # Reemplazo: la nueva generación sustituye a la anterior.
        poblacion = nueva_poblacion

    return mejor_ruta_global, mejor_distancia_global, pd.DataFrame(historial)


# --- Ejecución del AG ---
print("Iniciando Algoritmo Genético...")
print(f"Configuración: {TAM_POBLACION} individuos | {GENERACIONES} generaciones máx.")
print("-" * 65)

mejor_ruta, mejor_distancia, historial_df = ejecutar_ag()

print("-" * 65)
print(f"\nAlgoritmo finalizado.")
print(f"Mejor distancia encontrada: {mejor_distancia:.2f} m  ({mejor_distancia/1000:.3f} km)")

---
## CELDA 12 – Resultados y tabla comparativa

Se presenta la comparación entre una ruta inicial aleatoria y la ruta optimizada por el AG.

In [ ]:
# ============================================================
# CELDA 12 – RESULTADOS Y COMPARATIVA
# ============================================================

# Generar una ruta inicial aleatoria para comparar.
ruta_aleatoria    = crear_cromosoma()
dist_aleatoria    = distancia_total_ruta(ruta_aleatoria)
mejora_porcentaje = (dist_aleatoria - mejor_distancia) / dist_aleatoria * 100

# Construir la ruta completa con nombres para mostrar.
ruta_completa_ids   = [IDX_INICIO] + mejor_ruta + [IDX_FIN]
ruta_completa_nombres = [PUNTOS[i]["nombre"] for i in ruta_completa_ids]

print("=" * 65)
print(" RESULTADO DEL ALGORITMO GENÉTICO")
print("=" * 65)
print(f" Fuente de distancias : {FUENTE_DIST}")
print(f" Semilla              : {SEMILLA}")
print("-" * 65)
print(f" Ruta aleatoria (referencia): {dist_aleatoria:.2f} m")
print(f" Ruta optimizada (AG):        {mejor_distancia:.2f} m")
print(f" Mejora obtenida:             {mejora_porcentaje:.1f}%")
print("-" * 65)
print(" Orden de visita optimizado:")
for i, nombre in enumerate(ruta_completa_nombres):
    print(f"   Paso {i:>2}: {nombre}")
print("=" * 65)

# --- Tabla resumen ---
resumen_df = pd.DataFrame([
    {
        "Caso": "Ruta aleatoria (referencia)",
        "Cromosoma": str(ruta_aleatoria),
        "Distancia total (m)": round(dist_aleatoria, 2),
        "Distancia total (km)": round(dist_aleatoria / 1000, 3)
    },
    {
        "Caso": "Ruta optimizada por AG",
        "Cromosoma": str(mejor_ruta),
        "Distancia total (m)": round(mejor_distancia, 2),
        "Distancia total (km)": round(mejor_distancia / 1000, 3)
    }
])
display(resumen_df)

# --- Detalle tramo a tramo ---
print("\nDetalle de tramos (ruta optimizada):")
detalle = []
for k in range(len(ruta_completa_ids) - 1):
    orig = ruta_completa_ids[k]
    dest = ruta_completa_ids[k + 1]
    d = MATRIZ_DIST[orig][dest]
    detalle.append({
        "Tramo": f"{k+1}",
        "Origen": PUNTOS[orig]["nombre"],
        "Destino": PUNTOS[dest]["nombre"],
        "Distancia (m)": round(d, 1)
    })
detalle_df = pd.DataFrame(detalle)
display(detalle_df)

---
## CELDA 13 – Gráficos de evolución del AG

Se grafican las métricas de convergencia del algoritmo a lo largo de las generaciones.

In [ ]:
# ============================================================
# CELDA 13 – GRÁFICOS DE EVOLUCIÓN DEL ALGORITMO GENÉTICO
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Evolución de la mejor distancia por generación.
axes[0].plot(
    historial_df["Generacion"],
    historial_df["Mejor distancia (m)"],
    color="steelblue", linewidth=2, label="Mejor distancia"
)
axes[0].plot(
    historial_df["Generacion"],
    historial_df["Distancia promedio (m)"],
    color="orange", linewidth=1.5, linestyle="--", alpha=0.8, label="Distancia promedio"
)
axes[0].axhline(
    y=mejor_distancia, color="red", linestyle=":", linewidth=1.5,
    label=f"Mejor global = {mejor_distancia:.1f} m"
)
axes[0].set_title("Evolución de la distancia por generación", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Generación")
axes[0].set_ylabel("Distancia (m)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Evolución del fitness del mejor individuo.
axes[1].plot(
    historial_df["Generacion"],
    historial_df["Fitness mejor"],
    color="green", linewidth=2
)
axes[1].set_title("Evolución del fitness (mejor individuo)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Generación")
axes[1].set_ylabel("Fitness (1/distancia)")
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    "Algoritmo Genético – Recolección de Residuos Sólidos\nZona Los Sauces, Pimentel – Chiclayo",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("evolucion_ag.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado como: evolucion_ag.png")

---
## CELDA 14 – Mapa interactivo de la ruta optimizada

Se visualiza la ruta óptima obtenida sobre un mapa interactivo con **Folium**. Si OSMnx está disponible, los tramos siguen las calles reales; de lo contrario se trazan líneas directas entre puntos.

In [ ]:
# ============================================================
# CELDA 14 – MAPA INTERACTIVO DE LA RUTA OPTIMIZADA
# ============================================================

def obtener_tramo_osm(id_origen, id_destino):
    """
    Devuelve la lista de coordenadas (lat, lng) del camino más corto
    entre dos puntos usando la red vial de OSM.
    Si OSM no está disponible, devuelve la línea directa entre los puntos.
    """
    if G_osm is not None and nodos_snap is not None:
        try:
            nodo_o = nodos_snap[id_origen]
            nodo_d = nodos_snap[id_destino]
            camino = nx.shortest_path(G_osm, nodo_o, nodo_d, weight="length")
            return [(G_osm.nodes[n]["y"], G_osm.nodes[n]["x"]) for n in camino]
        except Exception:
            pass
    # Respaldo: línea directa.
    return [
        (PUNTOS[id_origen]["lat"],  PUNTOS[id_origen]["lng"]),
        (PUNTOS[id_destino]["lat"], PUNTOS[id_destino]["lng"])
    ]


def crear_mapa_ruta(ruta_ids):
    """
    Crea un mapa Folium interactivo con:
    - Marcadores numerados en cada punto de recolección.
    - Marcadores especiales (estrella) para inicio y fin.
    - Trazado de la ruta óptima sobre el mapa.
    """
    # Centro del mapa.
    lats = [PUNTOS[i]["lat"] for i in ruta_ids]
    lngs = [PUNTOS[i]["lng"] for i in ruta_ids]
    centro = (sum(lats) / len(lats), sum(lngs) / len(lngs))

    mapa = folium.Map(location=centro, zoom_start=16, tiles="OpenStreetMap")

    # Colores de los tramos (gradiente para mostrar el orden de la ruta).
    num_tramos = len(ruta_ids) - 1

    # Dibujar los tramos de la ruta.
    for k in range(num_tramos):
        orig = ruta_ids[k]
        dest = ruta_ids[k + 1]
        coords_tramo = obtener_tramo_osm(orig, dest)
        dist_tramo   = MATRIZ_DIST[orig][dest]

        folium.PolyLine(
            coords_tramo,
            weight=4,
            color="#1a73e8",
            opacity=0.85,
            tooltip=f"Tramo {k+1}: {PUNTOS[orig]['nombre']} → {PUNTOS[dest]['nombre']} | {dist_tramo:.0f} m"
        ).add_to(mapa)

    # Dibujar los marcadores de cada punto.
    for orden, pid in enumerate(ruta_ids):
        p = PUNTOS[pid]

        if pid == IDX_INICIO:
            icono  = folium.Icon(color="green", icon="play", prefix="fa")
            etiqueta = f"INICIO | {p['nombre']}"
        elif pid == IDX_FIN:
            icono  = folium.Icon(color="red", icon="flag", prefix="fa")
            etiqueta = f"FIN | {p['nombre']}"
        else:
            icono  = folium.Icon(color="blue", icon="trash", prefix="fa")
            etiqueta = f"Paso {orden}: {p['nombre']}"

        folium.Marker(
            location=[p["lat"], p["lng"]],
            popup=folium.Popup(etiqueta, max_width=200),
            tooltip=etiqueta,
            icon=icono
        ).add_to(mapa)

        # Número de orden junto al marcador.
        folium.Marker(
            location=[p["lat"] + 0.00005, p["lng"] + 0.00005],
            icon=folium.DivIcon(
                html=f'<div style="font-size:10px;font-weight:bold;color:#333">{orden}</div>',
                icon_size=(20, 20)
            )
        ).add_to(mapa)

    # Título del mapa.
    titulo_html = """
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                z-index: 1000; background: white; padding: 8px 16px;
                border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
                font-family: Arial; font-size: 13px; font-weight: bold;">
        Ruta Óptima – Recolección de Residuos Sólidos<br>
        <span style="font-size:11px; font-weight:normal;">
            Zona Los Sauces, Pimentel – Chiclayo | AG (OX1 + Swap + Inversión)
        </span>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(titulo_html))

    return mapa


# --- Crear y mostrar el mapa ---
ruta_completa_ids = [IDX_INICIO] + mejor_ruta + [IDX_FIN]
mapa_resultado    = crear_mapa_ruta(ruta_completa_ids)

mapa_resultado.save("mapa_ruta_residuos_sauces.html")
print("Mapa guardado como: mapa_ruta_residuos_sauces.html")

display(mapa_resultado)

---
## CELDA 15 – Resumen final para el informe

Resumen de todos los parámetros y resultados para incluir en el informe del trabajo de aplicación.

In [ ]:
# ============================================================
# CELDA 15 – RESUMEN FINAL PARA EL INFORME
# ============================================================

gens_ejecutadas = len(historial_df)
mejor_gen       = historial_df.loc[historial_df["Mejor distancia (m)"].idxmin(), "Generacion"]

print("=" * 65)
print(" RESUMEN PARA EL INFORME")
print(" Trabajo de Aplicación 02 – Algoritmos Genéticos")
print(" Desarrollo de Sistemas Inteligentes – USAT")
print("=" * 65)
print()
print(" CASO DE ESTUDIO")
print(f"   Zona: Los Sauces, Pimentel – Chiclayo, Perú")
print(f"   Puntos de recolección: {len(INTERMEDIOS)} (+ 1 inicio + 1 fin)")
print(f"   Fuente de distancias: {FUENTE_DIST}")
print()
print(" CONFIGURACIÓN DEL ALGORITMO GENÉTICO")
print(f"   Representación: permutación de ruta (path representation)")
print(f"   Cruce: Order Crossover OX1 (Davis, 1985)")
print(f"   Mutación: Swap + Simple Inversion Mutation (SIM)")
print(f"   Selección: Torneo de {TAM_TORNEO} individuos")
print(f"   Elitismo: {ELITISMO} individuos")
print(f"   Tamaño de población: {TAM_POBLACION}")
print(f"   Generaciones máx.: {GENERACIONES}")
print(f"   Prob. cruce: {PROB_CRUCE} | Prob. swap: {PROB_MUTACION_SWAP} | Prob. inv.: {PROB_MUTACION_INVERSION}")
print(f"   Semilla: {SEMILLA}")
print()
print(" RESULTADOS")
print(f"   Generaciones ejecutadas: {gens_ejecutadas}")
print(f"   Generación donde se halló la mejor ruta: {mejor_gen}")
print(f"   Distancia ruta aleatoria (referencia): {dist_aleatoria:.2f} m ({dist_aleatoria/1000:.3f} km)")
print(f"   Distancia ruta optimizada (AG):        {mejor_distancia:.2f} m ({mejor_distancia/1000:.3f} km)")
print(f"   Mejora obtenida:                       {mejora_porcentaje:.2f}%")
print()
print(" ORDEN DE VISITA ÓPTIMO")
for i, pid in enumerate(ruta_completa_ids):
    print(f"   {i:>2}. {PUNTOS[pid]['nombre']:<30}  "
          f"lat={PUNTOS[pid]['lat']:.7f}, lng={PUNTOS[pid]['lng']:.7f}")
print("=" * 65)

print("\nÚltimas 10 generaciones del historial:")
display(historial_df.tail(10))